# Importing Libraries

In [1]:
# Importing the other libraries

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import VarianceThreshold

In [ ]:
# Importing the dataset
df = pd.read_excel('Overall_Dataset.xlsx') #Change the name of the excel sheet

In [4]:
X = df.drop(labels= ['Name','Family','Activation Energy'], axis=1)
y = df['Activation Energy']


In [ ]:
# Descriptive statistics for each column
y.describe()
X.describe()

In [ ]:
# Splitting the dataset into the Training set and Test set

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.1, random_state = 0)
X_train.count()

In [ ]:
#Idenitfy corelated independent variables through heatmap
corrmat = X_train.corr()
fig, ax = plt.subplots()
fig.set_size_inches(8,8)
sns.heatmap(corrmat)


# Correlation Plot (Figure 3)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

corr = X_train.corr()
labels = corr.columns
n = len(labels)

# Mask ONLY upper triangle (keep diagonal)
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

# Extract indices
y, x = np.where(~mask)
values = corr.values[~mask]

# Bubble sizes
sizes = np.abs(values) * 900

# Plot
fig, ax = plt.subplots(figsize=(12, 12))

scatter = ax.scatter(
    x, y,
    s=sizes,
    c=values,
    cmap='viridis_r',   # reversed viridis
    vmin=-1, vmax=1,
    edgecolors='none'
)

# Axis formatting
ax.set_xticks(np.arange(n))
ax.set_yticks(np.arange(n))
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_yticklabels(labels)

ax.invert_yaxis()

# Align grid
ax.set_xlim(-0.5, n - 0.5)
ax.set_ylim(n - 0.5, -0.5)

# Colorbar
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Correlation')

plt.savefig(
    "correlation_bubble.png",
    dpi=600,
    bbox_inches='tight',
    facecolor='white'
)
plt.tight_layout()
plt.show()

# Fitting Multi-Linear Regression

In [ ]:
# Fitting Multi-Linear Regression
from sklearn.linear_model import LinearRegression
modelMLR = LinearRegression()

modelMLR.fit(X_train, y_train)

# Measure the R2 for training and test set
model_score = modelMLR.score(X_train, y_train)
print("The training R2 is: %.3f" % modelMLR.score(X_train, y_train))
print("The test R2 is: %.3f " % modelMLR.score(X_test, y_test))

from sklearn.metrics import mean_squared_error
from math import sqrt

y_predicted = modelMLR.predict(X_test)
y_predicted_train = modelMLR.predict(X_train)

# The mean squared error & Variance
print("MSE: %.2f" % mean_squared_error(y_test, y_predicted))
print("RMSE of test set: %.2f" % sqrt(mean_squared_error(y_test, y_predicted)))
print("RMSE of train set: %.2f" % sqrt(mean_squared_error(y_train, y_predicted_train)))

# k-cross validation
from sklearn.model_selection import cross_val_score
accuracies = cross_val_score(estimator=modelMLR, X=X_train, y=y_train, cv=5,scoring='neg_root_mean_squared_error')
print("The mean accuracy (CV RMSE) is: ", -accuracies.mean())

# Plotting the joint plot of actual v/s predicted
pp_tr = modelMLR.predict(X_train)

from sklearn.metrics import mean_squared_error, r2_score
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.ticker as ticker

# -----------------------------
# Global font control
# -----------------------------
fontsize = 18
fontname = 'Arial'

plt.rcParams.update({
    "font.family": fontname,
    "font.size": fontsize
})

# -----------------------------
# Joint plot
# -----------------------------
g = sns.JointGrid(x=y_test, y=y_predicted, height=6)

# Scatter
sns.scatterplot(x=y_train, y=pp_tr, s=100, color='orange', ax=g.ax_joint)
sns.scatterplot(x=y_test, y=y_predicted, s=100, color='blue', ax=g.ax_joint)

# Regression
sns.regplot(x=y_test, y=y_predicted, scatter=False, color='black', ax=g.ax_joint)

# Legend
#g.ax_joint.legend(["Train", "Test"], prop={'family': fontname, 'size': fontsize})

# -----------------------------
# Axis labels (same font + size)
# -----------------------------
ax = g.ax_joint
ax.set_xlabel("Actual", fontsize=fontsize, fontname=fontname)
ax.set_ylabel("Predicted", fontsize=fontsize, fontname=fontname)

# -----------------------------
# Marginals
# -----------------------------
sns.histplot(x=y_train, ax=g.ax_marg_x, color='orange', alpha=0.5)
sns.histplot(x=y_test, ax=g.ax_marg_x, color='blue', alpha=0.5)

sns.histplot(y=pp_tr, ax=g.ax_marg_y, color='orange', alpha=0.5)
sns.histplot(y=y_predicted, ax=g.ax_marg_y, color='blue', alpha=0.5)

# -----------------------------
# Ticks (major + minor SAME font + size)
# -----------------------------
ax.minorticks_on()

ax.xaxis.set_minor_locator(ticker.AutoMinorLocator(2))
ax.yaxis.set_minor_locator(ticker.AutoMinorLocator(2))

# Apply SAME font + size to ALL tick labels
for label in ax.get_xticklabels(which='both'):
    label.set_fontname(fontname)
    label.set_fontsize(fontsize)

for label in ax.get_yticklabels(which='both'):
    label.set_fontname(fontname)
    label.set_fontsize(fontsize)

# -----------------------------
# Control number of ticks
# -----------------------------
plt.locator_params(axis='both', nbins=5)

# -----------------------------
# Show
# -----------------------------
plt.show()

# Gaussian Process Regression

In [ ]:
# Fitting Gaussian Process Regression to the Training set with scaling
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C

# Scale X
sc_X = StandardScaler()
X_train_scaled = sc_X.fit_transform(X_train)
X_test_scaled = sc_X.transform(X_test)

# Scale y only if needed
# If y_train is a pandas Series, convert to numpy array if required
sc_y = StandardScaler()
y_train_scaled = sc_y.fit_transform(y_train.values.reshape(-1, 1)).ravel()
y_test_scaled = sc_y.transform(y_test.values.reshape(-1, 1)).ravel()

# Define kernel
kernel = C(1.0) * RBF(0.5)

# Build model
modelGPR = GaussianProcessRegressor(kernel=kernel, alpha=1e-1, random_state=0)

# Fit model on scaled X and scaled y
modelGPR.fit(X_train_scaled, y_train_scaled)

# Measure the R2 for training and test set
print("The training R2 is: %.3f" % modelGPR.score(X_train_scaled, y_train_scaled))
print("The test R2 is: %.3f " % modelGPR.score(X_test_scaled, y_test_scaled))

from sklearn.metrics import mean_squared_error
from math import sqrt

# Predict on scaled space
y_predicted_scaled = modelGPR.predict(X_test_scaled)
y_predicted_train_scaled = modelGPR.predict(X_train_scaled)

# Convert predictions back to original y scale
y_predicted = sc_y.inverse_transform(y_predicted_scaled.reshape(-1, 1)).ravel()
y_predicted_train = sc_y.inverse_transform(y_predicted_train_scaled.reshape(-1, 1)).ravel()

# The mean squared error & Variance on original scale
print("MSE: %.2f" % mean_squared_error(y_test, y_predicted))
print("RMSE of test set: %.2f" % sqrt(mean_squared_error(y_test, y_predicted)))
print("RMSE of train set: %.2f" % sqrt(mean_squared_error(y_train, y_predicted_train)))

# k-cross validation
from sklearn.model_selection import cross_val_score
accuracies = cross_val_score(estimator=modelGPR, X=X_train_scaled, y=y_train_scaled, cv=5,scoring='neg_root_mean_squared_error')
print("The mean accuracy (CV RMSE) is: ", -accuracies.mean())

# Plotting the joint plot of actual v/s predicted
pp_tr = y_predicted_train

from sklearn.metrics import mean_squared_error, r2_score
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.ticker as ticker

# -----------------------------
# Global font control
# -----------------------------
fontsize = 18
fontname = 'Arial'

plt.rcParams.update({
    "font.family": fontname,
    "font.size": fontsize
})

# -----------------------------
# Joint plot
# -----------------------------
g = sns.JointGrid(x=y_test, y=y_predicted, height=6)

# Scatter
sns.scatterplot(x=y_train, y=pp_tr, s=100, color='orange', ax=g.ax_joint)
sns.scatterplot(x=y_test, y=y_predicted, s=100, color='blue', ax=g.ax_joint)

# Regression
sns.regplot(x=y_test, y=y_predicted, scatter=False, color='black', ax=g.ax_joint)

# Legend
#g.ax_joint.legend(["Train", "Test"], prop={'family': fontname, 'size': fontsize})

# -----------------------------
# Axis labels (same font + size)
# -----------------------------
ax = g.ax_joint
ax.set_xlabel("Actual", fontsize=fontsize, fontname=fontname)
ax.set_ylabel("Predicted", fontsize=fontsize, fontname=fontname)

# -----------------------------
# Marginals
# -----------------------------
sns.histplot(x=y_train, ax=g.ax_marg_x, color='orange', alpha=0.5)
sns.histplot(x=y_test, ax=g.ax_marg_x, color='blue', alpha=0.5)

sns.histplot(y=pp_tr, ax=g.ax_marg_y, color='orange', alpha=0.5)
sns.histplot(y=y_predicted, ax=g.ax_marg_y, color='blue', alpha=0.5)

# -----------------------------
# Ticks (major + minor SAME font + size)
# -----------------------------
ax.minorticks_on()

ax.xaxis.set_minor_locator(ticker.AutoMinorLocator(2))
ax.yaxis.set_minor_locator(ticker.AutoMinorLocator(2))

# Apply SAME font + size to ALL tick labels
for label in ax.get_xticklabels(which='both'):
    label.set_fontname(fontname)
    label.set_fontsize(fontsize)

for label in ax.get_yticklabels(which='both'):
    label.set_fontname(fontname)
    label.set_fontsize(fontsize)

# -----------------------------
# Control number of ticks
# -----------------------------
plt.locator_params(axis='both', nbins=5)

# -----------------------------
# Show
# -----------------------------
plt.show()

# Random Forest Regression

In [ ]:
# Fitting RandomForest Regression to the Training set
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor()
model.fit(X_train, y_train)

# Measure the R2 for training and test set
model_score = model.score(X_train, y_train)
print("The training R2 is: ", model.score(X_train, y_train))
print("The test R2 is: ", model.score(X_test, y_test))

from sklearn.metrics import mean_squared_error
from math import sqrt

y_predicted = model.predict(X_test)
y_predicted_train = model.predict(X_train)

# The mean squared error & Variance
print("MSE: %.2f"% mean_squared_error(y_test, y_predicted))
print("RMSE of test set: %.2f"% sqrt(mean_squared_error(y_test, y_predicted)))
print("RMSE of train set: %.2f"% sqrt(mean_squared_error(y_train, y_predicted_train)))

# k-cross validation
from sklearn.model_selection import cross_val_score
accuracies = cross_val_score(estimator = model, X = X_train, y= y_train, cv=5,scoring='neg_root_mean_squared_error')
print("The mean accuracy (CV RMSE) is: ", -accuracies.mean())

# Plotting the joint plot of actual v/s predicted
pp_tr = model.predict(X_train)

from sklearn.metrics import mean_squared_error, r2_score
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.ticker as ticker

# -----------------------------
# Global font control
# -----------------------------
fontsize = 18
fontname = 'Arial'

plt.rcParams.update({
    "font.family": fontname,
    "font.size": fontsize
})

# -----------------------------
# Joint plot
# -----------------------------
g = sns.JointGrid(x=y_test, y=y_predicted, height=6)

# Scatter
sns.scatterplot(x=y_train, y=pp_tr, s=100, color='orange', ax=g.ax_joint)
sns.scatterplot(x=y_test, y=y_predicted, s=100, color='blue', ax=g.ax_joint)

# Regression
sns.regplot(x=y_test, y=y_predicted, scatter=False, color='black', ax=g.ax_joint)

# Legend
#g.ax_joint.legend(["Train", "Test"], prop={'family': fontname, 'size': fontsize})

# -----------------------------
# Axis labels (same font + size)
# -----------------------------
ax = g.ax_joint
ax.set_xlabel("Actual", fontsize=fontsize, fontname=fontname)
ax.set_ylabel("Predicted", fontsize=fontsize, fontname=fontname)

# -----------------------------
# Marginals
# -----------------------------
sns.histplot(x=y_train, ax=g.ax_marg_x, color='orange', alpha=0.5)
sns.histplot(x=y_test, ax=g.ax_marg_x, color='blue', alpha=0.5)

sns.histplot(y=pp_tr, ax=g.ax_marg_y, color='orange', alpha=0.5)
sns.histplot(y=y_predicted, ax=g.ax_marg_y, color='blue', alpha=0.5)

# -----------------------------
# Ticks (major + minor SAME font + size)
# -----------------------------
ax.minorticks_on()

ax.xaxis.set_minor_locator(ticker.AutoMinorLocator(2))
ax.yaxis.set_minor_locator(ticker.AutoMinorLocator(2))

# Apply SAME font + size to ALL tick labels
for label in ax.get_xticklabels(which='both'):
    label.set_fontname(fontname)
    label.set_fontsize(fontsize)

for label in ax.get_yticklabels(which='both'):
    label.set_fontname(fontname)
    label.set_fontsize(fontsize)

# -----------------------------
# Control number of ticks
# -----------------------------
plt.locator_params(axis='both', nbins=5)

# -----------------------------
# Show
# -----------------------------
plt.show()

# Support Vector Regression

In [ ]:
# Fitting Support Vector Regression to the Training set
from sklearn.svm import SVR

model = SVR()
model.fit(X_train, y_train.ravel())

# Measure the R2 for training and test set
model_score = model.score(X_train, y_train)
print("The training R2 is: ", model.score(X_train, y_train))
print("The test R2 is: ", model.score(X_test, y_test))

from sklearn.metrics import mean_squared_error
from math import sqrt

y_predicted = model.predict(X_test)
y_predicted_train = model.predict(X_train)

# The mean squared error & Variance
print("MSE: %.2f"% mean_squared_error(y_test, y_predicted))
print("RMSE of test set: %.2f"% sqrt(mean_squared_error(y_test, y_predicted)))
print("RMSE of train set: %.2f"% sqrt(mean_squared_error(y_train, y_predicted_train)))

# k-cross validation
from sklearn.model_selection import cross_val_score
accuracies = cross_val_score(estimator = model, X = X_train, y= y_train, cv=5,scoring='neg_root_mean_squared_error')
print("The mean accuracy (CV RMSE) is: ", -accuracies.mean())

# Plotting the joint plot of actual v/s predicted
pp_tr = model.predict(X_train)

from sklearn.metrics import mean_squared_error, r2_score
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.ticker as ticker

# -----------------------------
# Global font control
# -----------------------------
fontsize = 18
fontname = 'Arial'

plt.rcParams.update({
    "font.family": fontname,
    "font.size": fontsize
})

# -----------------------------
# Joint plot
# -----------------------------
g = sns.JointGrid(x=y_test, y=y_predicted, height=6)

# Scatter
sns.scatterplot(x=y_train, y=pp_tr, s=100, color='orange', ax=g.ax_joint)
sns.scatterplot(x=y_test, y=y_predicted, s=100, color='blue', ax=g.ax_joint)

# Regression
sns.regplot(x=y_test, y=y_predicted, scatter=False, color='black', ax=g.ax_joint)

# Legend
#g.ax_joint.legend(["Train", "Test"], prop={'family': fontname, 'size': fontsize})

# -----------------------------
# Axis labels (same font + size)
# -----------------------------
ax = g.ax_joint
ax.set_xlabel("Actual", fontsize=fontsize, fontname=fontname)
ax.set_ylabel("Predicted", fontsize=fontsize, fontname=fontname)

# -----------------------------
# Marginals
# -----------------------------
sns.histplot(x=y_train, ax=g.ax_marg_x, color='orange', alpha=0.5)
sns.histplot(x=y_test, ax=g.ax_marg_x, color='blue', alpha=0.5)

sns.histplot(y=pp_tr, ax=g.ax_marg_y, color='orange', alpha=0.5)
sns.histplot(y=y_predicted, ax=g.ax_marg_y, color='blue', alpha=0.5)

# -----------------------------
# Ticks (major + minor SAME font + size)
# -----------------------------
ax.minorticks_on()

ax.xaxis.set_minor_locator(ticker.AutoMinorLocator(2))
ax.yaxis.set_minor_locator(ticker.AutoMinorLocator(2))

# Apply SAME font + size to ALL tick labels
for label in ax.get_xticklabels(which='both'):
    label.set_fontname(fontname)
    label.set_fontsize(fontsize)

for label in ax.get_yticklabels(which='both'):
    label.set_fontname(fontname)
    label.set_fontsize(fontsize)

# -----------------------------
# Control number of ticks
# -----------------------------
plt.locator_params(axis='both', nbins=5)

# -----------------------------
# Show
# -----------------------------
plt.show()

# Gradient Boosting Regression

In [ ]:
# Fitting GBDT Regression to training data

from sklearn.ensemble import GradientBoostingRegressor
modelGBT =  GradientBoostingRegressor()

modelGBT.fit(X_train, y_train)

#Measure the R2 for training and test set
model_score = modelGBT.score(X_train,y_train)
print("The training R2 is: %.3f" % modelGBT.score(X_train, y_train))
print("The test R2 is: %.3f "% modelGBT.score(X_test, y_test))

from sklearn.metrics import mean_squared_error
from math import sqrt


y_predicted = modelGBT.predict(X_test)
y_predicted_train = modelGBT.predict(X_train)

# The mean squared error & Variance
print("MSE: %.2f"% mean_squared_error(y_test, y_predicted))
print("RMSE of test set: %.2f"% sqrt(mean_squared_error(y_test, y_predicted)))
print("RMSE of train set: %.2f"% sqrt(mean_squared_error(y_train, y_predicted_train)))


#k-cross validation
from sklearn.model_selection import cross_val_score
accuracies = cross_val_score(estimator = modelGBT, X = X_train, y= y_train, cv=5,scoring='neg_root_mean_squared_error')
print("The mean accuracy (CV RMSE) is: ", -accuracies.mean())


#Plotting the joint plot of  actual v/s predicted
pp_tr = modelGBT.predict(X_train)

from sklearn.metrics import mean_squared_error, r2_score
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.ticker as ticker

# -----------------------------
# Global font control
# -----------------------------
fontsize = 18
fontname = 'Arial'

plt.rcParams.update({
    "font.family": fontname,
    "font.size": fontsize
})

# -----------------------------
# Joint plot
# -----------------------------
g = sns.JointGrid(x=y_test, y=y_predicted, height=6)

# Scatter
sns.scatterplot(x=y_train, y=pp_tr, s=100, color='orange', ax=g.ax_joint)
sns.scatterplot(x=y_test, y=y_predicted, s=100, color='blue', ax=g.ax_joint)

# Regression
sns.regplot(x=y_test, y=y_predicted, scatter=False, color='black', ax=g.ax_joint)

# Legend
#g.ax_joint.legend(["Train", "Test"], prop={'family': fontname, 'size': fontsize})

# -----------------------------
# Axis labels (same font + size)
# -----------------------------
ax = g.ax_joint
ax.set_xlabel("Actual", fontsize=fontsize, fontname=fontname)
ax.set_ylabel("Predicted", fontsize=fontsize, fontname=fontname)

# -----------------------------
# Marginals
# -----------------------------
sns.histplot(x=y_train, ax=g.ax_marg_x, color='orange', alpha=0.5)
sns.histplot(x=y_test, ax=g.ax_marg_x, color='blue', alpha=0.5)

sns.histplot(y=pp_tr, ax=g.ax_marg_y, color='orange', alpha=0.5)
sns.histplot(y=y_predicted, ax=g.ax_marg_y, color='blue', alpha=0.5)

# -----------------------------
# Ticks (major + minor SAME font + size)
# -----------------------------
ax.minorticks_on()

ax.xaxis.set_minor_locator(ticker.AutoMinorLocator(2))
ax.yaxis.set_minor_locator(ticker.AutoMinorLocator(2))

# Apply SAME font + size to ALL tick labels
for label in ax.get_xticklabels(which='both'):
    label.set_fontname(fontname)
    label.set_fontsize(fontsize)

for label in ax.get_yticklabels(which='both'):
    label.set_fontname(fontname)
    label.set_fontsize(fontsize)

# -----------------------------
# Control number of ticks
# -----------------------------
plt.locator_params(axis='both', nbins=5)

# -----------------------------
# Show
# -----------------------------
plt.show()

# Hyper Parameter Tuning through GridSearchCV

In [ ]:
#Grid search for RF
from sklearn.model_selection import GridSearchCV
#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.1,random_state=0)
model = RandomForestRegressor()


param_grid={'n_estimators':np.arange(100,1001,100).tolist(),
                'max_depth':[40,80,120,160],
                'min_samples_split':[2,4,6,8,10],
                'min_samples_leaf': [1, 2, 4, 6],
                'max_features': ['sqrt', 'log2']}

tuning = GridSearchCV(estimator=model, 
                          param_grid = param_grid,
                          cv=5,
                          verbose=1,
                          n_jobs=-1)
    

tuning.fit(X_train, y_train)

print("Best parameters:")
print(tuning.best_params_)

In [ ]:
# Fitting optimized RandomForest Regression to the Training set
from sklearn.ensemble import RandomForestRegressor

modelRF = RandomForestRegressor(max_depth = 200, min_samples_leaf = 1, n_estimators = 100, min_samples_split= 4)
modelRF.fit(X_train, y_train)

# Measure the R2 for training and test set
model_score = modelRF.score(X_train, y_train)
print("The training R2 is: ", modelRF.score(X_train, y_train))
print("The test R2 is: ", modelRF.score(X_test, y_test))

from sklearn.metrics import mean_squared_error
from math import sqrt

y_predicted = modelRF.predict(X_test)
y_predicted_train = modelRF.predict(X_train)

# The mean squared error & Variance
print("MSE: %.2f"% mean_squared_error(y_test, y_predicted))
print("RMSE of test set: %.2f"% sqrt(mean_squared_error(y_test, y_predicted)))
print("RMSE of train set: %.2f"% sqrt(mean_squared_error(y_train, y_predicted_train)))

# k-cross validation
from sklearn.model_selection import cross_val_score
accuraciesRF = cross_val_score(estimator = modelRF, X = X_train, y= y_train, cv=5)
print("The mean accuracy is: ", accuraciesRF.mean())


cv_rmse = cross_val_score(
    estimator=modelRF,
    X=X_train,
    y=y_train,
    cv=5,
    scoring="neg_root_mean_squared_error"
)

print("CV RMSE for each fold:", -cv_rmse)
print("Mean CV RMSE:", -cv_rmse.mean())
print("Std CV RMSE:", cv_rmse.std())

import numpy as np

# -----------------------------
# User-controlled plotting vars
# -----------------------------
x_min, x_max = 0, 2.5
y_min, y_max = 0, 2.5
tick_interval = 0.5

train_color = "#4DB5C1"
test_color = "#F9AE47"
reg_line_color = 'black'

train_circle_size = 180
test_circle_size = 180

label_fontsize = 20
tick_fontsize = 20

# -----------------------------
# Predictions
# -----------------------------
pp_tr = modelRF.predict(X_train)
pp_te = modelRF.predict(X_test)

# -----------------------------
# Joint plot
# -----------------------------
g = sns.JointGrid(x=y_test, y=pp_te, height=8)

# Scatter points
sns.scatterplot(
    x=y_train, y=pp_tr,
    s=train_circle_size,
    color=train_color,
    edgecolor='black',
    ax=g.ax_joint,
    label='Train',
)

sns.scatterplot(
    x=y_test, y=pp_te,
    s=test_circle_size,
    color=test_color,
    edgecolor='black',
    ax=g.ax_joint,
    label='Test'
)

# Regression line for test set
sns.regplot(
    x=y_test, y=pp_te,
    scatter=False,
    color=reg_line_color,
    ax=g.ax_joint
)
g.ax_joint.legend(fontsize=20)

# 1:1 reference line
g.ax_joint.plot(
    [x_min, x_max], [y_min, y_max],
    linestyle='--', color='gray', linewidth=1.5
)

# Axis labels
#g.set_axis_labels("Actual", "Predicted", fontsize=label_fontsize, fontname='Arial')

# Marginal histogram
sns.histplot(x=y_train, ax=g.ax_marg_x, color=train_color, alpha=0.5)
sns.histplot(x=y_test, ax=g.ax_marg_x, color=test_color, alpha=0.5)

sns.histplot(y=pp_tr, ax=g.ax_marg_y, color=train_color, alpha=0.5)
sns.histplot(y=pp_te, ax=g.ax_marg_y, color=test_color, alpha=0.5)

# -----------------------------
# Axis limits and ticks
# -----------------------------
g.ax_joint.set_xlim(x_min, x_max)
g.ax_joint.set_ylim(y_min, y_max)

g.ax_joint.set_xticks(np.arange(x_min, x_max + tick_interval, tick_interval))
g.ax_joint.set_yticks(np.arange(y_min, y_max + tick_interval, tick_interval))
g.ax_joint.tick_params(axis='both',length=8,width=1.5)
# -----------------------------
# Make x and y axes intersect at 0
# while keeping full border visible
# -----------------------------
ax = g.ax_joint

# Move left and bottom spines to zero
ax.spines['left'].set_position(('data', 0))
ax.spines['bottom'].set_position(('data', 0))

# Keep full border
ax.spines['top'].set_visible(True)
ax.spines['right'].set_visible(True)

# Make all spines thicker if desired
for spine in ax.spines.values():
    spine.set_linewidth(1.5)

# Ensure ticks stay on bottom/left axes
ax.xaxis.set_ticks_position('bottom')
ax.yaxis.set_ticks_position('left')

# Tick font size
ax.tick_params(axis='both', labelsize=tick_fontsize)

# Legend
ax.legend()

plt.savefig("RF_model_full.svg", format="svg", bbox_inches="tight",dpi = 300)

plt.show()


# SHAP Analysis Full Feature Set

In [ ]:
import shap

# Create the explainer
explainer = shap.Explainer(modelRF)

# Note: using explainer, not explainer2 (which was undefined in your code)
shap_values = explainer(X_train)

# Display the bar plot
shap.plots.bar(shap_values, max_display=15)

# For newer SHAP versions, access the values attribute of the Explanation object
# and then calculate the mean absolute SHAP values
mean_abs_shap = np.abs(shap_values.values).mean(axis=0)

feature_names = X.columns.tolist()
target = y

shap_importance_df = pd.DataFrame({
    "Feature": feature_names,
    "MeanAbsSHAP": mean_abs_shap
}).sort_values("MeanAbsSHAP", ascending=False)

print("Full SHAP importance table:")
display(shap_importance_df)

## Recursive Model fitting 

In [33]:

# -----------------------------
# 3. Remove zero-impact features
# -----------------------------
shap_threshold = 0.005

nonzero_shap_df = shap_importance_df[shap_importance_df["MeanAbsSHAP"] > shap_threshold].copy()

# Keep top 8 among nonzero features
top_n = 8
extra_feature = 'Diameter (Å)'  # Changed to string instead of Series
top_features = nonzero_shap_df["Feature"].head(top_n).tolist()
top_features = list(dict.fromkeys(top_features + [extra_feature]))  # Put extra_feature in a list

print(f"Selected top {len(top_features)} nonzero-SHAP features for {target}:")
print(top_features)


Selected top 9 nonzero-SHAP features for [ 0  1  1  2  2  2  3  3  3  3  4  4  4  4  4  5  5  5  5  5  5  6  6  6
  6  6  6  6  7  7  7  7  7  7  7  7  8  8  8  8  8  8  8  8  8  9  9  9
  9  9  9  9  9  9  9 10 10 10 10 10 10 10 10 10 10 10 11 11 11 11 11 11
 11 11 11 11 11 11 12 12 12 12 12 12 12 12 12 12 12 12 12 13 13 13 13 13
 13 13 13 13 13 13 13 13 13 14 14 14 14 14 14 14 14 14 14 14 14 14 14 14]:
['RE', 'C bind', 'chi_avg', 'CE', 'Avg_C_metal_distance', 'ME', 'Avg NN Dist (Å)', 'GCN', 'Diameter (Å)']


# Repeat RF Model with Reduced Feature Set

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

# -----------------------------
# 4. Rebuild reduced datasets
# -----------------------------
X_train_top = X_train[top_features].copy()
X_test_top = X_test[top_features].copy()

# -----------------------------
# 5. Retrain on top 8 features
# -----------------------------

modelRF.fit(X_train_top, y_train)

# -----------------------------
# 6. Evaluate reduced model
# -----------------------------
y_pred_train_top = modelRF.predict(X_train_top)
y_pred_test_top = modelRF.predict(X_test_top)

train_rmse_top = sqrt(mean_squared_error(y_train, y_pred_train_top))
test_rmse_top = sqrt(mean_squared_error(y_test, y_pred_test_top))
train_r2_top = r2_score(y_train, y_pred_train_top)
test_r2_top = r2_score(y_test, y_pred_test_top)

# -------------------------
# 5-fold CV RMSE on retrained top-feature model
# -------------------------
cv_rmse_scores = cross_val_score(
    estimator=modelRF,
    X=X_train_top,
    y=y_train,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

cv_rmse_scores = -cv_rmse_scores
cv_rmse_mean = cv_rmse_scores.mean()
cv_rmse_std = cv_rmse_scores.std()


print(f"Train RMSE: {train_rmse_top:.3f}")
print(f"Test RMSE:  {test_rmse_top:.3f}") 
print(f"Train R2:   {train_r2_top:.3f}")
print(f"Test R2:    {test_r2_top:.3f}")
print(f"CV RMSE:    {cv_rmse_mean:.3f} ± {cv_rmse_std:.3f}")
print(f"Fold RMSEs: {np.round(cv_rmse_scores, 3)}")

In [ ]:
# -----------------------------
# User-controlled plotting vars
# -----------------------------
x_min, x_max = 0, 2.5
y_min, y_max = 0, 2.5
tick_interval = 0.5

train_color = "#4DB5C1"
test_color = "#F9AE47"
reg_line_color = 'black'

train_circle_size = 180
test_circle_size = 180

label_fontsize = 20
tick_fontsize = 20

# -----------------------------
# Predictions
# -----------------------------
pp_tr = modelRF.predict(X_train_top)
pp_te = modelRF.predict(X_test_top)

# -----------------------------
# Joint plot
# -----------------------------
g = sns.JointGrid(x=y_test, y=pp_te, height=8)

# Scatter points
sns.scatterplot(
    x=y_train, y=pp_tr,
    s=train_circle_size,
    color=train_color,
    edgecolor='black',
    ax=g.ax_joint,
    label='Train'
)

sns.scatterplot(
    x=y_test, y=pp_te,
    s=test_circle_size,
    color=test_color,
    edgecolor='black',
    ax=g.ax_joint,
    label='Test'
)

# Regression line for test set
sns.regplot(
    x=y_test, y=pp_te,
    scatter=False,
    color=reg_line_color,
    ax=g.ax_joint
)

g.ax_joint.legend(fontsize=20)

# 1:1 reference line
g.ax_joint.plot(
    [x_min, x_max], [y_min, y_max],
    linestyle='--', color='gray', linewidth=1.5
)

# Axis labels
#g.set_axis_labels("Actual", "Predicted", fontsize=label_fontsize, fontname='Arial')

# Marginal histogram
sns.histplot(x=y_train, ax=g.ax_marg_x, color=train_color, alpha=0.5)
sns.histplot(x=y_test, ax=g.ax_marg_x, color=test_color, alpha=0.5)

sns.histplot(y=pp_tr, ax=g.ax_marg_y, color=train_color, alpha=0.5)
sns.histplot(y=pp_te, ax=g.ax_marg_y, color=test_color, alpha=0.5)

# -----------------------------
# Axis limits and ticks
# -----------------------------
g.ax_joint.set_xlim(x_min, x_max)
g.ax_joint.set_ylim(y_min, y_max)

g.ax_joint.set_xticks(np.arange(x_min, x_max + tick_interval, tick_interval))
g.ax_joint.set_yticks(np.arange(y_min, y_max + tick_interval, tick_interval))
g.ax_joint.tick_params(axis='both',length=8,width=1.5)
# -----------------------------
# Make x and y axes intersect at 0
# while keeping full border visible
# -----------------------------
ax = g.ax_joint

# Move left and bottom spines to zero
ax.spines['left'].set_position(('data', 0))
ax.spines['bottom'].set_position(('data', 0))

# Keep full border
ax.spines['top'].set_visible(True)
ax.spines['right'].set_visible(True)

# Make all spines thicker if desired
for spine in ax.spines.values():
    spine.set_linewidth(1.5)

# Ensure ticks stay on bottom/left axes
ax.xaxis.set_ticks_position('bottom')
ax.yaxis.set_ticks_position('left')

# Tick font size
ax.tick_params(axis='both', labelsize=tick_fontsize)

# Legend
ax.legend()

plt.savefig("RF_model_recursive.svg", format="svg", bbox_inches="tight",dpi = 300)

plt.show()


# Reiteration of SHAP

In [ ]:
# Create the explainer
explainer_new = shap.Explainer(modelRF)

# Calculate SHAP values
shap_values_new = explainer_new(X_train_top)

# Display the bar plot
shap.plots.bar(shap_values_new, max_display=15)

# Calculate mean absolute SHAP values from the SHAP values we just computed
mean_abs_shap = np.abs(shap_values_new.values).mean(axis=0)

# Get feature names from X_train_top
top_features = X_train_top.columns.tolist()

# Ensure both arrays have the same length
# This is important because we're creating a DataFrame with these as columns
assert len(top_features) == len(mean_abs_shap), "Arrays must be of the same length"

# Create DataFrame with SHAP importance values
shap_importance_new = pd.DataFrame({
    "Feature": top_features,
    "MeanAbsSHAP": mean_abs_shap
}).sort_values("MeanAbsSHAP", ascending=False)

print("Full SHAP importance table:")
display(shap_importance_new)

# Fitting Multi-Linear Regression (After Reduction)

In [ ]:
# Fitting Multi-Linear Regression
from sklearn.linear_model import LinearRegression
modelMLR = LinearRegression()

modelMLR.fit(X_train_top, y_train)

# Measure the R2 for training and test set
model_score = modelMLR.score(X_train_top, y_train)
print("The training R2 is: %.3f" % modelMLR.score(X_train_top, y_train))
print("The test R2 is: %.3f " % modelMLR.score(X_test_top, y_test))

from sklearn.metrics import mean_squared_error
from math import sqrt

y_predicted = modelMLR.predict(X_test_top)
y_predicted_train = modelMLR.predict(X_train_top)

# The mean squared error & Variance
print("MSE: %.2f" % mean_squared_error(y_test, y_predicted))
print("RMSE of test set: %.2f" % sqrt(mean_squared_error(y_test, y_predicted)))
print("RMSE of train set: %.2f" % sqrt(mean_squared_error(y_train, y_predicted_train)))

# k-cross validation
from sklearn.model_selection import cross_val_score
accuracies = cross_val_score(estimator=modelMLR, X=X_train_top, y=y_train, cv=5,scoring='neg_root_mean_squared_error')
print("The mean accuracy (CV RMSE) is: ", -accuracies.mean())

# Plotting the joint plot of actual v/s predicted
pp_tr = modelMLR.predict(X_train_top)

from sklearn.metrics import mean_squared_error, r2_score
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.ticker as ticker

# -----------------------------
# Global font control
# -----------------------------
fontsize = 18
fontname = 'Arial'

plt.rcParams.update({
    "font.family": fontname,
    "font.size": fontsize
})

# -----------------------------
# Joint plot
# -----------------------------
g = sns.JointGrid(x=y_test, y=y_predicted, height=6)

# Scatter
sns.scatterplot(x=y_train, y=pp_tr, s=100, color='orange', ax=g.ax_joint)
sns.scatterplot(x=y_test, y=y_predicted, s=100, color='blue', ax=g.ax_joint)

# Regression
sns.regplot(x=y_test, y=y_predicted, scatter=False, color='black', ax=g.ax_joint)

# Legend
#g.ax_joint.legend(["Train", "Test"], prop={'family': fontname, 'size': fontsize})

# -----------------------------
# Axis labels (same font + size)
# -----------------------------
ax = g.ax_joint
ax.set_xlabel("Actual", fontsize=fontsize, fontname=fontname)
ax.set_ylabel("Predicted", fontsize=fontsize, fontname=fontname)

# -----------------------------
# Marginals
# -----------------------------
sns.histplot(x=y_train, ax=g.ax_marg_x, color='orange', alpha=0.5)
sns.histplot(x=y_test, ax=g.ax_marg_x, color='blue', alpha=0.5)

sns.histplot(y=pp_tr, ax=g.ax_marg_y, color='orange', alpha=0.5)
sns.histplot(y=y_predicted, ax=g.ax_marg_y, color='blue', alpha=0.5)

# -----------------------------
# Ticks (major + minor SAME font + size)
# -----------------------------
ax.minorticks_on()

ax.xaxis.set_minor_locator(ticker.AutoMinorLocator(2))
ax.yaxis.set_minor_locator(ticker.AutoMinorLocator(2))

# Apply SAME font + size to ALL tick labels
for label in ax.get_xticklabels(which='both'):
    label.set_fontname(fontname)
    label.set_fontsize(fontsize)

for label in ax.get_yticklabels(which='both'):
    label.set_fontname(fontname)
    label.set_fontsize(fontsize)

# -----------------------------
# Control number of ticks
# -----------------------------
plt.locator_params(axis='both', nbins=5)

# -----------------------------
# Show
# -----------------------------
plt.show()

# Fitting Gaussian Process Regression (After Reduction)

In [ ]:
# Fitting Gaussian Process Regression to the Training set with scaling
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C

# Scale X
sc_X = StandardScaler()
X_train_scaled = sc_X.fit_transform(X_train_top)
X_test_scaled = sc_X.transform(X_test_top)

# Scale y only if needed
# If y_train is a pandas Series, convert to numpy array if required
sc_y = StandardScaler()
y_train_scaled = sc_y.fit_transform(y_train.values.reshape(-1, 1)).ravel()
y_test_scaled = sc_y.transform(y_test.values.reshape(-1, 1)).ravel()

# Define kernel
kernel = C(1.0) * RBF(0.5)

# Build model
modelGPR = GaussianProcessRegressor(kernel=kernel, alpha=1e-1, random_state=0)

# Fit model on scaled X and scaled y
modelGPR.fit(X_train_scaled, y_train_scaled)

# Measure the R2 for training and test set
print("The training R2 is: %.3f" % modelGPR.score(X_train_scaled, y_train_scaled))
print("The test R2 is: %.3f " % modelGPR.score(X_test_scaled, y_test_scaled))

from sklearn.metrics import mean_squared_error
from math import sqrt

# Predict on scaled space
y_predicted_scaled = modelGPR.predict(X_test_scaled)
y_predicted_train_scaled = modelGPR.predict(X_train_scaled)

# Convert predictions back to original y scale
y_predicted = sc_y.inverse_transform(y_predicted_scaled.reshape(-1, 1)).ravel()
y_predicted_train = sc_y.inverse_transform(y_predicted_train_scaled.reshape(-1, 1)).ravel()

# The mean squared error & Variance on original scale
print("MSE: %.2f" % mean_squared_error(y_test, y_predicted))
print("RMSE of test set: %.2f" % sqrt(mean_squared_error(y_test, y_predicted)))
print("RMSE of train set: %.2f" % sqrt(mean_squared_error(y_train, y_predicted_train)))

# k-cross validation
from sklearn.model_selection import cross_val_score
accuracies = cross_val_score(estimator=modelGPR, X=X_train_scaled, y=y_train_scaled, cv=5)
print("The mean accuracy is: ", accuracies.mean())

# Plotting the joint plot of actual v/s predicted
pp_tr = y_predicted_train


from sklearn.metrics import mean_squared_error, r2_score
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.ticker as ticker

# -----------------------------
# Global font control
# -----------------------------
fontsize = 18
fontname = 'Arial'

plt.rcParams.update({
    "font.family": fontname,
    "font.size": fontsize
})

# -----------------------------
# Joint plot
# -----------------------------
g = sns.JointGrid(x=y_test, y=y_predicted, height=6)

# Scatter
sns.scatterplot(x=y_train, y=pp_tr, s=100, color='orange', ax=g.ax_joint)
sns.scatterplot(x=y_test, y=y_predicted, s=100, color='blue', ax=g.ax_joint)

# Regression
sns.regplot(x=y_test, y=y_predicted, scatter=False, color='black', ax=g.ax_joint)

# Legend
#g.ax_joint.legend(["Train", "Test"], prop={'family': fontname, 'size': fontsize})

# -----------------------------
# Axis labels (same font + size)
# -----------------------------
ax = g.ax_joint
ax.set_xlabel("Actual", fontsize=fontsize, fontname=fontname)
ax.set_ylabel("Predicted", fontsize=fontsize, fontname=fontname)

# -----------------------------
# Marginals
# -----------------------------
sns.histplot(x=y_train, ax=g.ax_marg_x, color='orange', alpha=0.5)
sns.histplot(x=y_test, ax=g.ax_marg_x, color='blue', alpha=0.5)

sns.histplot(y=pp_tr, ax=g.ax_marg_y, color='orange', alpha=0.5)
sns.histplot(y=y_predicted, ax=g.ax_marg_y, color='blue', alpha=0.5)

# -----------------------------
# Ticks (major + minor SAME font + size)
# -----------------------------
ax.minorticks_on()

ax.xaxis.set_minor_locator(ticker.AutoMinorLocator(2))
ax.yaxis.set_minor_locator(ticker.AutoMinorLocator(2))

# Apply SAME font + size to ALL tick labels
for label in ax.get_xticklabels(which='both'):
    label.set_fontname(fontname)
    label.set_fontsize(fontsize)

for label in ax.get_yticklabels(which='both'):
    label.set_fontname(fontname)
    label.set_fontsize(fontsize)

# -----------------------------
# Control number of ticks
# -----------------------------
plt.locator_params(axis='both', nbins=5)

# -----------------------------
# Show
# -----------------------------
plt.show()

# Fitting Support Vector Regression (After Reduction)

In [ ]:
# Fitting Support Vector Regression to the Training set
from sklearn.svm import SVR

model = SVR()
model.fit(X_train_top, y_train.ravel())

# Measure the R2 for training and test set
model_score = model.score(X_train_top, y_train)
print("The training R2 is: %.3f" % model.score(X_train_top, y_train))
print("The test R2 is: %.3f" %  model.score(X_test_top, y_test))

from sklearn.metrics import mean_squared_error
from math import sqrt

y_predicted = model.predict(X_test_top)
y_predicted_train = model.predict(X_train_top)

# The mean squared error & Variance
print("MSE: %.2f"% mean_squared_error(y_test, y_predicted))
print("RMSE of test set: %.2f"% sqrt(mean_squared_error(y_test, y_predicted)))
print("RMSE of train set: %.2f"% sqrt(mean_squared_error(y_train, y_predicted_train)))

# k-cross validation
from sklearn.model_selection import cross_val_score
accuracies = cross_val_score(estimator = model, X = X_train_top, y= y_train, cv=5,scoring='neg_root_mean_squared_error')
print("The mean accuracy (CV RMSE) is: ", -accuracies.mean())

# Plotting the joint plot of actual v/s predicted
pp_tr = model.predict(X_train_top)

from sklearn.metrics import mean_squared_error, r2_score
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.ticker as ticker

# -----------------------------
# Global font control
# -----------------------------
fontsize = 18
fontname = 'Arial'

plt.rcParams.update({
    "font.family": fontname,
    "font.size": fontsize
})

# -----------------------------
# Joint plot
# -----------------------------
g = sns.JointGrid(x=y_test, y=y_predicted, height=6)

# Scatter
sns.scatterplot(x=y_train, y=pp_tr, s=100, color='orange', ax=g.ax_joint)
sns.scatterplot(x=y_test, y=y_predicted, s=100, color='blue', ax=g.ax_joint)

# Regression
sns.regplot(x=y_test, y=y_predicted, scatter=False, color='black', ax=g.ax_joint)

# Legend
#g.ax_joint.legend(["Train", "Test"], prop={'family': fontname, 'size': fontsize})

# -----------------------------
# Axis labels (same font + size)
# -----------------------------
ax = g.ax_joint
ax.set_xlabel("Actual", fontsize=fontsize, fontname=fontname)
ax.set_ylabel("Predicted", fontsize=fontsize, fontname=fontname)

# -----------------------------
# Marginals
# -----------------------------
sns.histplot(x=y_train, ax=g.ax_marg_x, color='orange', alpha=0.5)
sns.histplot(x=y_test, ax=g.ax_marg_x, color='blue', alpha=0.5)

sns.histplot(y=pp_tr, ax=g.ax_marg_y, color='orange', alpha=0.5)
sns.histplot(y=y_predicted, ax=g.ax_marg_y, color='blue', alpha=0.5)

# -----------------------------
# Ticks (major + minor SAME font + size)
# -----------------------------
ax.minorticks_on()

ax.xaxis.set_minor_locator(ticker.AutoMinorLocator(2))
ax.yaxis.set_minor_locator(ticker.AutoMinorLocator(2))

# Apply SAME font + size to ALL tick labels
for label in ax.get_xticklabels(which='both'):
    label.set_fontname(fontname)
    label.set_fontsize(fontsize)

for label in ax.get_yticklabels(which='both'):
    label.set_fontname(fontname)
    label.set_fontsize(fontsize)

# -----------------------------
# Control number of ticks
# -----------------------------
plt.locator_params(axis='both', nbins=5)

# -----------------------------
# Show
# -----------------------------
plt.show()

# Fitting GBDT Regression (After Reduction)

In [ ]:
# Fitting GBDT Regression to training data

from sklearn.ensemble import GradientBoostingRegressor
modelGBT =  GradientBoostingRegressor()

modelGBT.fit(X_train_top, y_train)

#Measure the R2 for training and test set
model_score = modelGBT.score(X_train_top,y_train)
print("The training R2 is: %.3f" % modelGBT.score(X_train_top, y_train))
print("The test R2 is: %.3f "% modelGBT.score(X_test_top, y_test))

from sklearn.metrics import mean_squared_error
from math import sqrt


y_predicted = modelGBT.predict(X_test_top)
y_predicted_train = modelGBT.predict(X_train_top)

# The mean squared error & Variance
print("MSE: %.2f"% mean_squared_error(y_test, y_predicted))
print("RMSE of test set: %.2f"% sqrt(mean_squared_error(y_test, y_predicted)))
print("RMSE of train set: %.2f"% sqrt(mean_squared_error(y_train, y_predicted_train)))


#k-cross validation
from sklearn.model_selection import cross_val_score
accuracies = cross_val_score(estimator = modelGBT, X = X_train_top, y= y_train, cv=5,scoring='neg_root_mean_squared_error')
print("The mean accuracy (CV RMSE) is: ", -accuracies.mean())


#Plotting the joint plot of  actual v/s predicted
pp_tr = modelGBT.predict(X_train_top)

from sklearn.metrics import mean_squared_error, r2_score
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.ticker as ticker

# -----------------------------
# Global font control
# -----------------------------
fontsize = 18
fontname = 'Arial'

plt.rcParams.update({
    "font.family": fontname,
    "font.size": fontsize
})

# -----------------------------
# Joint plot
# -----------------------------
g = sns.JointGrid(x=y_test, y=y_predicted, height=6)

# Scatter
sns.scatterplot(x=y_train, y=pp_tr, s=100, color='orange', ax=g.ax_joint)
sns.scatterplot(x=y_test, y=y_predicted, s=100, color='blue', ax=g.ax_joint)

# Regression
sns.regplot(x=y_test, y=y_predicted, scatter=False, color='black', ax=g.ax_joint)

# Legend
#g.ax_joint.legend(["Train", "Test"], prop={'family': fontname, 'size': fontsize})

# -----------------------------
# Axis labels (same font + size)
# -----------------------------
ax = g.ax_joint
ax.set_xlabel("Actual", fontsize=fontsize, fontname=fontname)
ax.set_ylabel("Predicted", fontsize=fontsize, fontname=fontname)

# -----------------------------
# Marginals
# -----------------------------
sns.histplot(x=y_train, ax=g.ax_marg_x, color='orange', alpha=0.5)
sns.histplot(x=y_test, ax=g.ax_marg_x, color='blue', alpha=0.5)

sns.histplot(y=pp_tr, ax=g.ax_marg_y, color='orange', alpha=0.5)
sns.histplot(y=y_predicted, ax=g.ax_marg_y, color='blue', alpha=0.5)

# -----------------------------
# Ticks (major + minor SAME font + size)
# -----------------------------
ax.minorticks_on()

ax.xaxis.set_minor_locator(ticker.AutoMinorLocator(2))
ax.yaxis.set_minor_locator(ticker.AutoMinorLocator(2))

# Apply SAME font + size to ALL tick labels
for label in ax.get_xticklabels(which='both'):
    label.set_fontname(fontname)
    label.set_fontsize(fontsize)

for label in ax.get_yticklabels(which='both'):
    label.set_fontname(fontname)
    label.set_fontsize(fontsize)

# -----------------------------
# Control number of ticks
# -----------------------------
plt.locator_params(axis='both', nbins=5)

# -----------------------------
# Show
# -----------------------------
plt.show()

# Leave-One-Family-Out validation for Random Forest

In [ ]:
# Leave-One-Family-Out validation for Random Forest
# Assumes your full dataset is in a pandas DataFrame called df
# Target column: 'Activation Energy'
# Group column: 'Family'

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import r2_score, mean_squared_error
from math import sqrt

# -----------------------------
# Define X, y, and groups
# -----------------------------
X = df[top_features].copy()
y = y
groups = df["Family"]




# Leave-One-Group-Out CV
# -----------------------------
logo = LeaveOneGroupOut()

results = []
y_true_all = []
y_pred_all = []

for fold, (train_idx, test_idx) in enumerate(logo.split(X, y, groups), start=1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    family_test = groups.iloc[test_idx].iloc[0]

    # Fit model
    modelRF.fit(X_train, y_train)

    # Predict
    y_pred_train = modelRF.predict(X_train)
    y_pred_test = modelRF.predict(X_test)

    # Metrics
    train_r2 = r2_score(y_train, y_pred_train)
    test_r2 = r2_score(y_test, y_pred_test)
    test_mse = mean_squared_error(y_test, y_pred_test)
    test_rmse = sqrt(test_mse)

    results.append({
        'Fold': fold,
        'Left_out_family': family_test,
        'Train_size': len(train_idx),
        'Test_size': len(test_idx),
        'Train_R2': train_r2,
        'Test_R2': test_r2,
        'Test_MSE': test_mse,
        'Test_RMSE': test_rmse
    })

    y_true_all.extend(y_test.tolist())
    y_pred_all.extend(y_pred_test.tolist())

    print(f"Fold {fold} | Left-out Family: {family_test}")
    print("Train R2: %.3f" % train_r2)
    print("Test R2: %.3f" % test_r2)
    print("Test RMSE: %.3f" % test_rmse)
    print("-" * 40)

# -----------------------------
# Summary table
# -----------------------------
results_df = pd.DataFrame(results)
print("\nPer-family results:")
print(results_df)

print("\nAverage performance across folds:")
print("Mean Train R2: %.3f" % results_df['Train_R2'].mean())
print("Mean Test R2: %.3f" % results_df['Test_R2'].mean())
print("Mean Test RMSE: %.3f" % results_df['Test_RMSE'].mean())

# Overall pooled metrics across all left-out predictions
overall_r2 = r2_score(y_true_all, y_pred_all)
overall_rmse = sqrt(mean_squared_error(y_true_all, y_pred_all))

print("\nOverall pooled performance:")
print("Overall Test R2: %.3f" % overall_r2)
print("Overall Test RMSE: %.3f" % overall_rmse)